# Waste Type Identification — Fase 2: confronto fra backbone (ricetta `geo_rrc`, batch effettivo 64)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BACKBONE = 'convnext_tiny'                                                      # 'resnet18', 'regnet_y_1_6gf', 'efficientnet_v2_s', 'convnext_tiny'

# Percorsi
BASE             = Path('/content/drive/MyDrive/2026_MLinf_gr41/Waste-Project')
DRIVE_DATASET_DIR = BASE / 'dataset'
DATASET_DIR      = Path('/content/dataset_local')
SPLIT_CSV        = BASE / 'splits' / 'split.csv'
MODELS_DIR       = BASE / 'models'
RESULTS_DIR      = BASE / 'results'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Iperparametri
IMG_SIZE        = 224
EFFECTIVE_BATCH = 64                                                            # batch effettivo uguale per tutti i backbone
NUM_EPOCHS      = 15
MOMENTUM        = 0.9
WEIGHT_DECAY    = 1e-4
NUM_CLASSES     = 8
SEED            = 1234
USE_AMP         = False

CLASS_NAMES = ['battery','clothing','glass','metal','organic','papery','plastic','undifferentiated']

BACKBONE_CONFIGS = {
    'resnet18':          {'save_tag': 'resnet18_geo_rrc',      'optimizer': 'sgd',   'lr': 0.01,  'micro_batch': 64},
    'regnet_y_1_6gf':    {'save_tag': 'regnety16gf_geo_rrc',   'optimizer': 'sgd',   'lr': 0.01,  'micro_batch': 32},
    'efficientnet_v2_s': {'save_tag': 'effv2s_geo_rrc',        'optimizer': 'sgd',   'lr': 0.01,  'micro_batch': 16},
    'convnext_tiny':     {'save_tag': 'convnext_tiny_geo_rrc', 'optimizer': 'adamw', 'lr': 5e-4, 'weight_decay': 0.05, 'micro_batch': 32},
}
assert BACKBONE in BACKBONE_CONFIGS, f'BACKBONE non valido: {BACKBONE}'
CFG = BACKBONE_CONFIGS[BACKBONE]
SAVE_TAG    = CFG['save_tag']
MICRO_BATCH = CFG['micro_batch']
assert EFFECTIVE_BATCH % MICRO_BATCH == 0, 'EFFECTIVE_BATCH deve essere multiplo di micro_batch'
ACCUM_STEPS = EFFECTIVE_BATCH // MICRO_BATCH
print(f'Backbone: {BACKBONE} | tag: {SAVE_TAG} | opt: {CFG["optimizer"]} | lr: {CFG["lr"]}')
print(f'micro-batch {MICRO_BATCH} x accumulo {ACCUM_STEPS} = batch effettivo {EFFECTIVE_BATCH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Backbone: convnext_tiny | tag: convnext_tiny_geo_rrc | opt: adamw | lr: 0.0005
micro-batch 32 x accumulo 2 = batch effettivo 64


In [ ]:
import time, random, copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import balanced_accuracy_score, recall_score
import matplotlib.pyplot as plt

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


## Copia locale del dataset

In [ ]:
import shutil
if not DATASET_DIR.exists():
    print('Copio il dataset in locale, attendi qualche minuto...')
    t0 = time.time(); shutil.copytree(DRIVE_DATASET_DIR, DATASET_DIR)
    print(f'Copia completata in {time.time() - t0:.0f}s')
else:
    print('Copia locale gia presente:', DATASET_DIR)

Copia locale gia presente: /content/dataset_local


## Dataset, preprocessing e ricetta `geo_rrc`

- **Train (`geo_rrc`)**: RandomResizedCrop(224, scale=(0.5,1.0)) + light (flip, piccola rotazione,
  ColorJitter blando). Il DataLoader usa il **micro-batch** fisico; l'accumulo ricostruisce il batch 64.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(3/4, 4/3)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class WasteDataset(Dataset):
    def __init__(self, df, root, transform):
        self.df = df.reset_index(drop=True); self.root = Path(root); self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(self.root / row['filepath']).convert('RGB')
        return self.transform(img), int(row['label'])

df = pd.read_csv(SPLIT_CSV)
train_df = df[df['split'] == 'train'].copy()
val_df   = df[df['split'] == 'val'].copy()
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

train_ds = WasteDataset(train_df, DATASET_DIR, train_tf)
val_ds   = WasteDataset(val_df,   DATASET_DIR, val_tf)

# batch_size del DataLoader = micro-batch fisico
train_loader = DataLoader(train_ds, batch_size=MICRO_BATCH, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=MICRO_BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

Train: 12413 | Val: 3102


## Costruzione di modello e ottimizzatore

In [ ]:
def build_model(name):
    if name == 'resnet18':
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'regnet_y_1_6gf':
        m = models.regnet_y_1_6gf(weights=models.RegNet_Y_1_6GF_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == 'efficientnet_v2_s':
        m = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif name == 'convnext_tiny':
        m = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        m.classifier[2] = nn.Linear(m.classifier[2].in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m

def build_optimizer(model, cfg):
    if cfg['optimizer'] == 'adamw':
        return torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg.get('weight_decay', 0.05))
    return torch.optim.SGD(model.parameters(), lr=cfg['lr'], momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)

model = build_model(BACKBONE).to(device)
optimizer = build_optimizer(model, CFG)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
criterion = nn.CrossEntropyLoss()
print(f'{BACKBONE}: {sum(p.numel() for p in model.parameters())/1e6:.2f}M parametri')

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 126MB/s] 


convnext_tiny: 27.83M parametri


## Training e validazione (con gradient accumulation)

Lo step dell'ottimizzatore avviene ogni `ACCUM_STEPS` micro-batch (e a fine epoca per svuotare
l'eventuale resto), così il batch effettivo è 64. La loss è divisa per `ACCUM_STEPS` perché i
gradienti si sommano. Si tiene il **miglior checkpoint** sulla balanced accuracy di validation, e
dopo l'epoca 1 si stampa il **picco di VRAM** per verificare il vincolo < 5 GB.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps = [], []
    for x, y in tqdm(loader, desc='val', leave=False):
        x = x.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(x)
        ps.append(out.argmax(1).cpu().numpy()); ys.append(y.numpy())
    y_true = np.concatenate(ys); y_pred = np.concatenate(ps)
    return balanced_accuracy_score(y_true, y_pred), y_true, y_pred

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
if device.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()

history = []; best_bal = -1.0; best_state = None; best_eval = None
n_batches = len(train_loader)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    running = 0.0; nb = 0
    optimizer.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(tqdm(train_loader, desc=f'epoch {epoch}/{NUM_EPOCHS} [{SAVE_TAG}]')):
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = model(x); loss = criterion(out, y) / ACCUM_STEPS
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == n_batches:
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running += loss.item() * ACCUM_STEPS; nb += 1
    scheduler.step()
    train_loss = running / max(nb, 1)

    val_bal, y_true, y_pred = evaluate(model, val_loader)
    history.append({'epoch': epoch, 'train_loss': train_loss, 'val_balacc': val_bal})
    print(f'epoch {epoch:02d} | train_loss {train_loss:.4f} | val_balAcc {val_bal:.4f}')

    if epoch == 1 and device.type == 'cuda':
        peak = torch.cuda.max_memory_allocated() / 1e9
        print(f'  >> picco VRAM training (epoca 1): {peak:.2f} GB  (vincolo < 5 GB)')

    if val_bal > best_bal:
        best_bal = val_bal; best_state = copy.deepcopy(model.state_dict()); best_eval = (y_true, y_pred)

print(f'\nMigliore val balanced accuracy: {best_bal:.4f}')

/tmp/ipykernel_727/3478801625.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


epoch 1/15 [convnext_tiny_geo_rrc]:   0%|          | 0/388 [00:00<?, ?it/s]

/tmp/ipykernel_727/3478801625.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


val:   0%|          | 0/97 [00:00<?, ?it/s]

/tmp/ipykernel_727/3478801625.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


...

epoch 15/15 [convnext_tiny_geo_rrc]:   0%|          | 0/388 [00:00<?, ?it/s]

Exception ignored in:     <function _MultiProcessingDataLoaderIter.__del__ at 0x7be1de407a60>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
    if w.is_alive():
    Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7be1de407a60> 
 Traceback (most recent call last):
^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
^^    ^^self._shutdown_workers()
^^  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
^^    ^if w.is_alive():
^ ^
   File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
       assert self._parent_pid == os.getpid(), 'can only test a child process'  
  ^^ ^^  ^^ ^ ^ ^^  ^^ 
^  File 

val:   0%|          | 0/97 [00:00<?, ?it/s]

/tmp/ipykernel_727/3478801625.py:6: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


epoch 15 | train_loss 0.0042 | val_balAcc 0.9756

Migliore val balanced accuracy: 0.9771


## Salvataggio pesi, curve e TPR per-classe

In [ ]:
weights_path = MODELS_DIR / f'{SAVE_TAG}.pth'
torch.save(best_state, weights_path)
print('Pesi salvati in:', weights_path)

hist_df = pd.DataFrame(history)
hist_df.to_csv(RESULTS_DIR / f'history_{SAVE_TAG}.csv', index=False)

fig, ax1 = plt.subplots(figsize=(6,4))
ax1.plot(hist_df['epoch'], hist_df['train_loss'], 'C0-o', label='train loss')
ax1.set_xlabel('epoch'); ax1.set_ylabel('train loss', color='C0')
ax2 = ax1.twinx()
ax2.plot(hist_df['epoch'], hist_df['val_balacc'], 'C1-s', label='val balAcc')
ax2.set_ylabel('val balanced accuracy', color='C1')
plt.title(f'{SAVE_TAG}  (best {best_bal:.4f})'); fig.tight_layout()
fig.savefig(RESULTS_DIR / f'curves_{SAVE_TAG}.png', dpi=120); plt.show()

y_true, y_pred = best_eval
tpr = recall_score(y_true, y_pred, average=None, labels=list(range(NUM_CLASSES)), zero_division=0)
tpr_df = pd.DataFrame({'class': CLASS_NAMES, 'label': list(range(NUM_CLASSES)), 'tpr_cleanval': tpr})
tpr_df.to_csv(RESULTS_DIR / f'cleanval_per_class_tpr_{SAVE_TAG}.csv', index=False)
print(tpr_df.to_string(index=False))